<a href="https://colab.research.google.com/github/Hari-HK23/ML-Final-Lab-Group-05/blob/main/notebooks/Member4_ML_Enigneer_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pickle
import pandas as pd
import numpy as np
import time
import logging
from pathlib import Path

In [3]:
from google.colab import files

uploaded = files.upload()

Saving best_fraud_detection_model.pkl to best_fraud_detection_model.pkl
Saving cleaned_fraud_data (10).csv to cleaned_fraud_data (10).csv


In [4]:
import os

print(os.listdir("/content"))

['.config', 'cleaned_fraud_data (10).csv', 'best_fraud_detection_model.pkl', 'sample_data']


In [5]:
MODEL_PATH = "/content/best_fraud_detection_model.pkl"
INPUT_PATH = "/content/cleaned_fraud_data (10).csv"

OUTPUT_PATH = "/content/prediction_results.csv"
LOG_PATH = "/content/inference.log"

In [6]:
with open(MODEL_PATH, "rb") as file:
    model = pickle.load(file)

print("✓ Model loaded successfully")
print("Model type:", type(model).__name__)

✓ Model loaded successfully
Model type: XGBClassifier


In [7]:
data = pd.read_csv(INPUT_PATH)

print("✓ Dataset loaded successfully")
print("Rows:", len(data))
print("Columns:", len(data.columns))

display(data.head())

✓ Dataset loaded successfully
Rows: 88235
Columns: 21


,city_pop,is_fraud,age,trans_hour,trans_dayofweek,distance_km,distance,log_amt,category_food_dining,category_gas_transport,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,93193,0,25,17,5,59.091675,0.535738,2.628285,False,False,...,False,False,False,False,False,False,False,False,False,False
1,27971,0,35,23,2,53.923925,0.567878,1.137833,False,False,...,False,False,False,False,False,False,False,True,False,False
2,123,0,47,18,6,33.631403,0.354923,4.236856,False,False,...,False,False,False,False,False,False,False,False,False,False
3,895,0,20,3,1,16.994574,0.164607,4.007151,False,False,...,True,False,False,False,False,False,False,False,False,False
4,11256,0,47,21,6,42.952314,0.403650,4.825189,False,False,...,False,False,False,False,False,False,False,True,False,False


In [8]:
print("Dataset columns:")

for column in data.columns:
    print("-", column)

Dataset columns:
- city_pop
- is_fraud
- age
- trans_hour
- trans_dayofweek
- distance_km
- distance
- log_amt
- category_food_dining
- category_gas_transport
- category_grocery_net
- category_grocery_pos
- category_health_fitness
- category_home
- category_kids_pets
- category_misc_net
- category_misc_pos
- category_personal_care
- category_shopping_net
- category_shopping_pos
- category_travel


In [9]:
data["late_night"] = data["trans_hour"].isin(
    [22, 23, 0, 1, 2, 3]
)

print("✓ late_night feature created")

display(
    data[
        ["trans_hour", "late_night"]
    ].head(10)
)

✓ late_night feature created


,trans_hour,late_night
0,17,False
1,23,True
2,18,False
3,3,True
4,21,False
5,1,True
6,7,False
7,5,False
8,3,True
9,10,False


In [10]:
high_risk_categories = [
    "category_shopping_net",
    "category_grocery_pos",
    "category_misc_net"
]

available_high_risk = [
    column
    for column in high_risk_categories
    if column in data.columns
]

print("Available high-risk categories:")
print(available_high_risk)


if available_high_risk:

    data["late_night_high_risk"] = (
        data["late_night"]
        & data[available_high_risk]
        .astype(bool)
        .any(axis=1)
    )

else:

    data["late_night_high_risk"] = False


print("\n✓ late_night_high_risk feature created")

display(
    data[
        [
            "late_night",
            "late_night_high_risk"
        ]
    ].head(10)
)

Available high-risk categories:
['category_shopping_net', 'category_grocery_pos', 'category_misc_net']

✓ late_night_high_risk feature created


,late_night,late_night_high_risk
0,False,False
1,True,True
2,False,False
3,True,True
4,False,False
5,True,False
6,False,False
7,False,False
8,True,False
9,False,False


In [16]:
expected_features = list(model.feature_names_in_)

print("Features expected by the trained model:")
print()

for i, feature in enumerate(expected_features, start=1):
    print(f"{i}. {feature}")

print("\nTotal features expected:", len(expected_features))

Features expected by the trained model:

1. log_amt
2. trans_hour
3. late_night
4. late_night_high_risk
5. age
6. category_food_dining
7. category_gas_transport
8. category_grocery_net
9. category_grocery_pos
10. category_health_fitness
11. category_home
12. category_kids_pets
13. category_misc_net
14. category_misc_pos
15. category_personal_care
16. category_shopping_net
17. category_shopping_pos
18. category_travel

Total features expected: 18


In [17]:
missing_features = [
    feature
    for feature in expected_features
    if feature not in data.columns
]

if missing_features:
    print("Missing features:")
    for feature in missing_features:
        print("-", feature)

    raise ValueError(
        "The dataset is missing features required by the model."
    )

print("✓ All 18 model features are available")

✓ All 18 model features are available


In [18]:
features = data[expected_features].copy()

# ------------------------------------------------------------
# Numeric features
# ------------------------------------------------------------

numeric_features = [
    "log_amt",
    "trans_hour",
    "age"
]

for column in numeric_features:
    features[column] = pd.to_numeric(
        features[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# Boolean / category features
# ------------------------------------------------------------

boolean_features = [
    "late_night",
    "late_night_high_risk"
]

boolean_features += [
    feature
    for feature in expected_features
    if feature.startswith("category_")
]

for column in boolean_features:
    if column in features.columns:
        features[column] = (
            features[column]
            .astype(bool)
        )


print("✓ All model features prepared")
print("\nFeature shape:", features.shape)

display(features.head())

✓ All model features prepared

Feature shape: (88235, 18)


,log_amt,trans_hour,late_night,late_night_high_risk,age,category_food_dining,category_gas_transport,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,2.628285,17,False,False,25,False,False,False,False,False,False,False,False,False,False,False,False,False
1,1.137833,23,True,True,35,False,False,False,False,False,False,False,False,False,False,True,False,False
2,4.236856,18,False,False,47,False,False,False,False,False,False,False,False,False,False,False,False,False
3,4.007151,3,True,True,20,False,False,False,True,False,False,False,False,False,False,False,False,False
4,4.825189,21,False,False,47,False,False,False,False,False,False,False,False,False,False,True,False,False


In [19]:
missing_values = features.isnull().sum()

print("Missing values:")

if missing_values.sum() == 0:
    print("✓ No missing values")
else:
    print(
        missing_values[
            missing_values > 0
        ]
    )

    raise ValueError(
        "Missing values detected in model input."
    )

Missing values:
✓ No missing values


In [20]:
print("Running PayShield fraud detection...")

start_time = time.perf_counter()

predictions = model.predict(features)

probabilities = model.predict_proba(features)

end_time = time.perf_counter()

print("✓ Prediction completed successfully")

Running PayShield fraud detection...
✓ Prediction completed successfully


In [21]:
total_latency_ms = (
    end_time - start_time
) * 1000

average_latency_ms = (
    total_latency_ms / len(features)
)

print(
    f"Total inference time: "
    f"{total_latency_ms:.4f} ms"
)

print(
    f"Average inference time per transaction: "
    f"{average_latency_ms:.6f} ms"
)

Total inference time: 1532.3314 ms
Average inference time per transaction: 0.017366 ms


In [22]:
classes = list(model.classes_)

print("Model classes:", classes)

if 0 not in classes or 1 not in classes:
    raise ValueError(
        "Model must contain classes 0 and 1."
    )

legitimate_index = classes.index(0)
fraud_index = classes.index(1)

legitimate_probability = (
    probabilities[:, legitimate_index]
)

fraud_probability = (
    probabilities[:, fraud_index]
)

print("✓ Fraud and legitimate probabilities calculated")

Model classes: [np.int64(0), np.int64(1)]
✓ Fraud and legitimate probabilities calculated


In [23]:
prediction_confidence = np.maximum(
    legitimate_probability,
    fraud_probability
)

print("✓ Prediction confidence calculated")

✓ Prediction confidence calculated


In [24]:
results = pd.DataFrame()

# Transaction ID
results["transaction_id"] = data.index

# Actual fraud label
results["actual_fraud"] = (
    data["is_fraud"].astype(int)
)

# Model prediction
results["predicted_fraud"] = (
    predictions.astype(int)
)

# Human-readable label
results["prediction_label"] = np.where(
    predictions.astype(int) == 1,
    "FRAUD",
    "LEGITIMATE"
)

# Confidence
results["prediction_confidence"] = (
    prediction_confidence
)

# Fraud probability
results["fraud_probability"] = (
    fraud_probability
)

# Legitimate probability
results["legitimate_probability"] = (
    legitimate_probability
)

# Average inference latency
results["inference_time_ms"] = (
    average_latency_ms
)

display(results.head(20))

,transaction_id,actual_fraud,predicted_fraud,prediction_label,prediction_confidence,fraud_probability,legitimate_probability,inference_time_ms
0,0,0,0,LEGITIMATE,0.998698,1.302358e-03,0.998698,0.017366
1,1,0,0,LEGITIMATE,0.999995,4.788464e-06,0.999995,0.017366
2,2,0,0,LEGITIMATE,0.999999,8.093475e-07,0.999999,0.017366
3,3,0,0,LEGITIMATE,0.999996,4.122912e-06,0.999996,0.017366
4,4,0,0,LEGITIMATE,0.999997,2.528108e-06,0.999997,0.017366
5,5,0,0,LEGITIMATE,0.999985,1.476832e-05,0.999985,0.017366
6,6,0,0,LEGITIMATE,0.999970,3.046830e-05,0.999970,0.017366
7,7,0,0,LEGITIMATE,0.999985,1.460458e-05,0.999985,0.017366
8,8,0,0,LEGITIMATE,0.999976,2.406258e-05,0.999976,0.017366
9,9,0,0,LEGITIMATE,0.999847,1.530115e-04,0.999847,0.017366


In [25]:
results.to_csv(
    OUTPUT_PATH,
    index=False
)

print("✓ prediction_results.csv created")
print("Saved to:", OUTPUT_PATH)

✓ prediction_results.csv created
Saved to: /content/prediction_results.csv


In [26]:
print("Total transactions:", len(results))

print(
    "Actual fraud transactions:",
    results["actual_fraud"].sum()
)

print(
    "Predicted fraud transactions:",
    results["predicted_fraud"].sum()
)

print(
    "Predicted legitimate transactions:",
    (
        results["predicted_fraud"] == 0
    ).sum()
)

display(results.head(20))

Total transactions: 88235
Actual fraud transactions: 150
Predicted fraud transactions: 116
Predicted legitimate transactions: 88119


,transaction_id,actual_fraud,predicted_fraud,prediction_label,prediction_confidence,fraud_probability,legitimate_probability,inference_time_ms
0,0,0,0,LEGITIMATE,0.998698,1.302358e-03,0.998698,0.017366
1,1,0,0,LEGITIMATE,0.999995,4.788464e-06,0.999995,0.017366
2,2,0,0,LEGITIMATE,0.999999,8.093475e-07,0.999999,0.017366
3,3,0,0,LEGITIMATE,0.999996,4.122912e-06,0.999996,0.017366
4,4,0,0,LEGITIMATE,0.999997,2.528108e-06,0.999997,0.017366
5,5,0,0,LEGITIMATE,0.999985,1.476832e-05,0.999985,0.017366
6,6,0,0,LEGITIMATE,0.999970,3.046830e-05,0.999970,0.017366
7,7,0,0,LEGITIMATE,0.999985,1.460458e-05,0.999985,0.017366
8,8,0,0,LEGITIMATE,0.999976,2.406258e-05,0.999976,0.017366
9,9,0,0,LEGITIMATE,0.999847,1.530115e-04,0.999847,0.017366


In [27]:
print("=" * 60)
print("PAYSHIELD SANITY CHECKS")
print("=" * 60)

# 1. Model
assert model is not None
print("✓ Model loaded")

# 2. Prediction count
assert len(predictions) == len(data)
print("✓ Prediction count matches dataset")

# 3. Binary predictions
assert set(
    np.unique(predictions)
).issubset({0, 1})

print("✓ Predictions are binary")

# 4. Probability range
assert np.all(
    (fraud_probability >= 0)
    & (fraud_probability <= 1)
)

assert np.all(
    (legitimate_probability >= 0)
    & (legitimate_probability <= 1)
)

print("✓ Probabilities are between 0 and 1")

# 5. Probability sum
assert np.allclose(
    fraud_probability + legitimate_probability,
    1.0,
    atol=1e-6
)

print("✓ Probabilities sum to 1")

# 6. Output file
assert Path(OUTPUT_PATH).exists()
print("✓ prediction_results.csv exists")

# 7. Row count
saved_results = pd.read_csv(OUTPUT_PATH)

assert len(saved_results) == len(data)
print("✓ Output row count matches input")

# 8. Latency
assert total_latency_ms >= 0
print("✓ Inference latency measured")

print("=" * 60)
print("ALL SANITY CHECKS PASSED ✓")
print("=" * 60)

PAYSHIELD SANITY CHECKS
✓ Model loaded
✓ Prediction count matches dataset
✓ Predictions are binary
✓ Probabilities are between 0 and 1
✓ Probabilities sum to 1
✓ prediction_results.csv exists
✓ Output row count matches input
✓ Inference latency measured
ALL SANITY CHECKS PASSED ✓


In [28]:
from google.colab import files

files.download(
    "/content/prediction_results.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>